In [ ]:
%%capture
import sys
!{sys.executable} -m pip install indiafactorlibrary

In [ ]:
# Notebook: 01_factor_zoo
# Series:   pyIndiaFactorInvesting
# Data:     Invespar Indian Factor Library
# Run in:   Colab / Binder / local
# Author:   Rajan Raju

# The Factor Zoo: What Moves Returns Beyond the Market?

The five puzzles in `00_indian_market_puzzles.ipynb` share a common feature: market returns alone cannot explain them. This notebook builds the vocabulary to say why — and shows what the Indian data actually looks like.

**Who this is for:** A working finance professional with undergraduate training. No equations you cannot follow. No black boxes.

**What you will leave with:**
- A working definition of a factor and why it should exist
- An intuition for why CAPM fails Indian data
- A first look at the Indian factor premia

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm

from src.data import load_nifty_sample, load_invespar_factors, align_data
from src.factors import compute_factor_summary

NIFTY_LABELS = {
    'nifty100': 'N100',                        'nifty_midcap150': 'MidCap150',
    'nifty_smallcap250': 'SmallCap250',         'nifty500_value50': 'Value50',
    'nifty100_quality30': 'Quality30',          'nifty_midcap150_quality50': 'MidQual50',
    'nifty_smallcap250_quality50': 'SmQual50',  'nifty200_momentum30': 'Momentum30',
    'nifty_midcap150_momentum50': 'MidMom50',
}

---
## Section 1 — The Single-Factor Baseline

The **Capital Asset Pricing Model (CAPM)** makes one central claim: the only risk that commands a return premium is *market risk* — the degree to which an asset moves with the overall market.

Its two assumptions matter for what comes next:
1. Investors are rational and hold the same diversified portfolio
2. All other risks can be diversified away — so no other risk is priced

**The CAPM equation:**

$$E[R_i] - R_f = \beta_i \cdot (E[R_m] - R_f)$$

*In plain English: your expected excess return depends only on how much your investment moves with the market — nothing else.*

The coefficient β (beta) measures that sensitivity. β = 1 means your investment moves one-for-one with the market. β = 1.5 means it is 50% more volatile.

**If CAPM is correct**, every asset should sit exactly on the Security Market Line (SML) — beta on the x-axis, expected excess return on the y-axis. The chart below tests that.

In [ ]:
factors = load_invespar_factors()
nifty = load_nifty_sample()
nifty_al, fac_al = align_data(nifty, factors[['MKT', 'RF']])

# MKT in the Invespar library is already the market excess return (MKT - RF)
betas, mean_exc = {}, {}
for col in nifty_al.columns:
    exc = nifty_al[col] - fac_al['RF']
    X = sm.add_constant(fac_al['MKT'])
    betas[col] = sm.OLS(exc, X).fit().params['MKT']
    mean_exc[col] = exc.mean()

In [ ]:
b_arr = list(betas.values())
sml_x = np.linspace(min(b_arr) - 0.1, max(b_arr) + 0.2, 100)
sml_y = fac_al['MKT'].mean() * sml_x

fig, ax = plt.subplots(figsize=(8, 5), dpi=120)
for col in nifty_al.columns:
    ax.scatter(betas[col], mean_exc[col], s=70, zorder=5)
    ax.annotate(NIFTY_LABELS.get(col, col), (betas[col], mean_exc[col]),
                fontsize=7, xytext=(4, 2), textcoords='offset points')
ax.plot(sml_x, sml_y, '--', color='black', lw=1.2, label='Theoretical SML (CAPM)')
ax.axhline(0, color='grey', lw=0.4, ls=':')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=2))
ax.set_xlabel('Beta (sensitivity to market excess return)', fontsize=10)
ax.set_ylabel('Mean Monthly Excess Return', fontsize=10)
ax.set_title('Security Market Line: Indian Index Evidence', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, frameon=False)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
ax.annotate('Source: Invespar Indian Factor Library', xy=(1, 0), xycoords='axes fraction',
            ha='right', va='top', fontsize=8, color='grey', xytext=(0, -28), textcoords='offset points')
fig.tight_layout()
plt.show()

The nine Nifty indices do not sit on the theoretical SML. Some (momentum, quality) have higher excess returns than CAPM predicts for their beta. Others sit below the line.

**The pattern of deviations is not random — it is systematic.** Momentum, quality, and size indices cluster in ways that CAPM has no language to describe.

*For the full regression treatment — t-statistics, F-tests, and model diagnostics on three specific Nifty indices — see `intro_to_ap.ipynb`.*

---
## Section 2 — What the Residuals Reveal

Each of the five puzzles from `00` corresponds to a systematic deviation that CAPM
cannot explain. These deviations have names:

| Puzzle from `00` | What CAPM misses |
|------------------|------------------|
| Size reversal (2018) | Size premium — **SMB** (Small Minus Big) |
| Quality in crisis | Profitability premium — **RMW** (Robust Minus Weak) |
| Momentum crash | Momentum factor — **WML** (Winners Minus Losers) |
| Fund dispersion | Combination of factor tilts — different betas to the same factors |
| Vol ≠ return | Low-risk anomaly — **BAB** (Betting Against Beta) |

Each factor name follows the long-short convention: long the desired characteristic,
short the opposite. SMB is long small-cap, short large-cap; WML is long recent winners,
short recent losers. Construction details are in `02_factor_construction.ipynb`.

> **Note on BAB:** The Betting Against Beta anomaly is documented in global markets
> but falls outside the Invespar Factor Library used in this series. It is listed here
> for completeness; it is not tested in the premia chart below.

Naming a deviation is not explaining it. The question is whether these patterns
are *persistent* and *systematic* enough to qualify as factors — or whether they
are coincidences in the data.

---
## Section 3 — Factor Premia: What the Indian Data Shows

A **factor** is a systematic, persistent return driver not explained by market exposure alone.

Three candidate explanations for why factor premia persist:
- **Risk:** investors demand a premium for bearing a type of risk that cannot be diversified away
- **Behaviour:** persistent investor mistakes — overreaction, anchoring — create predictable mispricings
- **Structure:** market frictions (liquidity constraints, institutional mandates) prevent arbitrageurs from eliminating the gap

All three may be partially true. The evidence favours different explanations for different factors.

The chart below shows the annualised mean return for each factor in the Indian market, with ±1 standard error bars. Wide bars mean the estimate is uncertain.

In [ ]:
fac_cols = ['MKT', 'SMB5', 'HML', 'RMW', 'WML']
summary = compute_factor_summary(factors[fac_cols])
n_per = factors[fac_cols].count()
se = (factors[fac_cols].std() / np.sqrt(n_per)) * 12
means = summary['Ann. Mean'].sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5), dpi=120)
colors = ['#2ca02c' if v > 0 else '#d62728' for v in means]
ax.barh(means.index, means.values, color=colors, alpha=0.75)
for i, idx in enumerate(means.index):
    ax.errorbar(means[idx], i, xerr=se[idx], fmt='none',
                color='black', capsize=4, lw=1.2)
ax.axvline(0, color='black', lw=0.8)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Annualised Mean Return', fontsize=10)
ax.set_title('Factor Premia: Indian Evidence', fontsize=11, fontweight='bold')
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
ax.annotate('Source: Invespar Indian Factor Library', xy=(1, 0), xycoords='axes fraction',
            ha='right', va='top', fontsize=8, color='grey', xytext=(0, -28), textcoords='offset points')
fig.tight_layout()
plt.show()

**Reading the chart:**
- A positive bar means the factor earned a premium on average in India
- Error bars show ±1 standard error of the annualised mean — if a bar does not clearly exceed its own error bar, treat the evidence as weak
- **WML (momentum)** and **MKT** have the largest and most robust premia — consistent with Raju (2022)
- **SMB5 (size)** is small and uncertain — consistent with the size reversal puzzle in `00`
- **RMW (profitability)** and **HML (value)** sit between them

*Factor construction — how these are actually built from Indian stock data — is covered in `02_factor_construction.ipynb`.*

---
## Section 4 — An Honest Caveat

Hundreds of factors have been published in academic literature. Most fail
out-of-sample: they worked in the data used to discover them, then stopped.
This is the **factor zoo problem**.

This series uses only factors with published Indian evidence — specifically
Raju (2022) on the Fama-French 4-, 5-, and 6-factor (FF4/FF5/FF6) framework
in India, and Agarwalla, Jacob and Varma (2013) on the Indian three-factor model.

These are not the only factors that matter in India. But they are the ones we
can defend with published evidence.

*"A factor that cannot be explained is not a factor — it is a coincidence."*

## What We Established

| | |
|---|---|
| **CAPM** | Useful baseline — explains most of the variance in Indian index returns |
| **Its failure** | Systematic deviations that follow factor patterns, not random noise |
| **Factors** | Named, persistent sources of those deviations — SMB, HML, RMW, WML |
| **Indian evidence** | Factor premia exist — but differ in magnitude from US benchmarks |

---

**Next:** `02_factor_construction.ipynb` — how SMB, HML, and WML are built from Indian stock data using the Invespar Factor Library.

**Going deeper:** `intro_to_ap.ipynb` — full regression mechanics, model diagnostics, and worked examples on three Nifty indices.